In [0]:
# ============================================================
# NOTEBOOK 2 — SILVER LAYER (Clean, Join, Enrich)
# Reads from : brazilian-ecommerce.bronze.*
# Writes to  : brazilian-ecommerce.silver.*
# ============================================================

CATALOG       = "brazilian-ecommerce"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SILVER_SCHEMA}`")

print(f"Bronze : {CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver : {CATALOG}.{SILVER_SCHEMA}")

In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, when, lit, trim, upper,
    datediff, unix_timestamp, round as spark_round,
    count, sum as spark_sum, avg, max as spark_max,
    row_number, rank, dense_rank, lag,
    current_timestamp, coalesce, isnan
)
from pyspark.sql.window import Window

# Read Bronze tables
df_orders    = spark.table(f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`orders`")
df_items     = spark.table(f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`order_items`")
df_customers = spark.table(f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`customers`")
df_products  = spark.table(f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`products`")
df_payments  = spark.table(f"`{CATALOG}`.`{BRONZE_SCHEMA}`.`payments`")

print("Bronze tables loaded.")
print(f"  orders    : {df_orders.count():,}")
print(f"  items     : {df_items.count():,}")
print(f"  customers : {df_customers.count():,}")
print(f"  products  : {df_products.count():,}")
print(f"  payments  : {df_payments.count():,}")

In [0]:
# Bronze stores all timestamps as strings — cast them properly in Silver
df_orders_clean = (
    df_orders

    # --- Drop Bronze audit columns (not needed in Silver) ---
    .drop("_ingested_at", "_source_file", "_layer")

    # --- Cast all timestamp strings to proper TimestampType ---
    .withColumn("order_purchase_timestamp",
                to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("order_approved_at",
                to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("order_delivered_carrier_date",
                to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("order_delivered_customer_date",
                to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("order_estimated_delivery_date",
                to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss"))

    # --- Derived columns (business logic) ---
    # Days taken to actually deliver
    .withColumn("actual_delivery_days",
                datediff(
                    col("order_delivered_customer_date"),
                    col("order_purchase_timestamp")
                ))
    # Days estimated at time of order
    .withColumn("estimated_delivery_days",
                datediff(
                    col("order_estimated_delivery_date"),
                    col("order_purchase_timestamp")
                ))
    # Was the delivery late? (key business KPI)
    .withColumn("is_late_delivery",
                when(
                    col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
                    lit(True)
                ).otherwise(lit(False)))

    # --- Standardise status to uppercase ---
    .withColumn("order_status", upper(trim(col("order_status"))))

    # --- Drop rows with no order_id (PK — must exist) ---
    .filter(col("order_id").isNotNull())

    # --- Deduplicate on primary key ---
    .dropDuplicates(["order_id"])
)

print(f"Orders after cleaning : {df_orders_clean.count():,}")
df_orders_clean.printSchema()

In [0]:
# --- Clean order items ---
df_items_clean = (
    df_items
    .drop("_ingested_at", "_source_file", "_layer")
    .withColumn("shipping_limit_date",
                to_timestamp(col("shipping_limit_date"), "yyyy-MM-dd HH:mm:ss"))
    # Total value per line item
    .withColumn("line_total",
                spark_round(col("price") + col("freight_value"), 2))
    # Drop items with null price (unusable for revenue analysis)
    .filter(col("price").isNotNull() & (col("price") > 0))
    .dropDuplicates(["order_id", "order_item_id"])   # composite PK
)

print(f"Items after cleaning : {df_items_clean.count():,}")

# --- Aggregate payments to order level ---
# One order can have multiple payment rows (installments, multiple methods)
df_payments_agg = (
    df_payments
    .drop("_ingested_at", "_source_file", "_layer")
    .groupBy("order_id")
    .agg(
        spark_round(spark_sum("payment_value"), 2).alias("total_payment_value"),
        spark_max("payment_installments").alias("max_installments"),
        count("payment_sequential").alias("payment_rows")
    )
)

print(f"Payments aggregated  : {df_payments_agg.count():,} orders")

In [0]:
# --- Clean customers ---
df_customers_clean = (
    df_customers
    .drop("_ingested_at", "_source_file", "_layer")
    .withColumn("customer_state", upper(trim(col("customer_state"))))
    .withColumn("customer_city",  trim(col("customer_city")))
    .filter(col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

print(f"Customers after cleaning : {df_customers_clean.count():,}")

# --- Clean products ---
df_products_clean = (
    df_products
    .drop("_ingested_at", "_source_file", "_layer")
    # Fill missing category with 'unknown'
    .withColumn("product_category_name",
                coalesce(col("product_category_name"), lit("unknown")))
    .filter(col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
)

print(f"Products after cleaning  : {df_products_clean.count():,}")

In [0]:
# This is the core Silver table — one row per order_item enriched with
# order, customer, product, and payment context

df_silver_master = (
    df_items_clean
    .alias("i")

    # Join orders (inner — items must have a valid order)
    .join(df_orders_clean.alias("o"),
          on="order_id",
          how="inner")

    # Join customers (left — keep order even if customer lookup fails)
    .join(df_customers_clean.alias("c"),
          on="customer_id",
          how="left")

    # Join products (left — keep item even if product lookup fails)
    .join(df_products_clean.alias("p"),
          on="product_id",
          how="left")

    # Join aggregated payments (left — keep order even if payment missing)
    .join(df_payments_agg.alias("pay"),
          on="order_id",
          how="left")

    # Select final columns — explicit selection, no SELECT *
    .select(
        # Order identifiers
        col("order_id"),
        col("order_item_id"),
        col("order_status"),
        col("order_purchase_timestamp"),
        col("order_delivered_customer_date"),

        # Delivery metrics
        col("actual_delivery_days"),
        col("estimated_delivery_days"),
        col("is_late_delivery"),

        # Item financials
        col("i.product_id"),
        col("i.seller_id"),
        col("i.price"),
        col("i.freight_value"),
        col("i.line_total"),

        # Customer
        col("customer_id"),
        col("c.customer_unique_id"),
        col("c.customer_city"),
        col("c.customer_state"),

        # Product
        col("p.product_category_name"),
        col("p.product_weight_g"),

        # Payment
        col("pay.total_payment_value"),
        col("pay.max_installments"),

        # Audit
        current_timestamp().alias("_silver_processed_at")
    )
)

print(f"Silver master rows : {df_silver_master.count():,}")
df_silver_master.printSchema()

In [0]:
# Window functions are among the most-asked Spark interview topics

# --- Window 1: Rank orders per customer by purchase date ---
w_customer = Window.partitionBy("customer_unique_id") \
                   .orderBy(col("order_purchase_timestamp").asc())

# --- Window 2: Rank products by revenue within each category ---
w_category = Window.partitionBy("product_category_name") \
                   .orderBy(col("line_total").desc())

df_silver_enriched = (
    df_silver_master

    # Customer order sequence — is this their 1st, 2nd, 3rd order?
    .withColumn("customer_order_rank",
                row_number().over(w_customer))

    # Is this a repeat customer?
    .withColumn("is_repeat_customer",
                when(col("customer_order_rank") > 1, True).otherwise(False))

    # Rank of this item's price within its product category
    .withColumn("price_rank_in_category",
                rank().over(w_category))
)

print(f"Enriched rows : {df_silver_enriched.count():,}")

# Preview window function results
display(
    df_silver_enriched
    .select("customer_unique_id", "order_id", "order_purchase_timestamp",
            "customer_order_rank", "is_repeat_customer")
    .filter(col("customer_order_rank") <= 3)
    .orderBy("customer_unique_id", "customer_order_rank")
    .limit(20)
)

In [0]:
SILVER_TABLE = f"`{CATALOG}`.`{SILVER_SCHEMA}`.`orders_master`"

(
    df_silver_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    # Partition by year+month — queries filtering on date will skip irrelevant partitions
    .partitionBy("customer_state")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table written → {SILVER_TABLE}")
spark.sql(f"SELECT COUNT(*) as total FROM {SILVER_TABLE}").show()

In [0]:
# OPTIMIZE compacts small files into larger ones (default 1GB target)
# ZORDER BY co-locates related data physically — speeds up filters on that column
spark.sql(f"""
    OPTIMIZE {SILVER_TABLE}
    ZORDER BY (order_purchase_timestamp, product_category_name)
""")

print("OPTIMIZE + ZORDER complete.")

# Check table history after optimize
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}") \
     .select("version", "timestamp", "operation", "operationParameters") \
     .show(5, truncate=False)

In [0]:
from pyspark.sql.functions import count, when

df_check = spark.table(SILVER_TABLE)
total = df_check.count()

checks = df_check.agg(
    count(when(col("order_id").isNull(),            1)).alias("null_order_ids"),
    count(when(col("customer_id").isNull(),         1)).alias("null_customer_ids"),
    count(when(col("price") <= 0,                   1)).alias("invalid_prices"),
    count(when(col("is_late_delivery") == True,     1)).alias("late_deliveries"),
    count(when(col("is_repeat_customer") == True,   1)).alias("repeat_customers"),
    count(when(col("product_category_name") == "unknown", 1)).alias("unknown_category"),
).collect()[0]

print(f"\n=== Silver Data Quality Report ===")
print(f"  Total rows          : {total:,}")
print(f"  Null order_ids      : {checks['null_order_ids']:,}")
print(f"  Null customer_ids   : {checks['null_customer_ids']:,}")
print(f"  Invalid prices      : {checks['invalid_prices']:,}")
print(f"  Late deliveries     : {checks['late_deliveries']:,}  ({checks['late_deliveries']/total*100:.1f}%)")
print(f"  Repeat customers    : {checks['repeat_customers']:,}  ({checks['repeat_customers']/total*100:.1f}%)")
print(f"  Unknown category    : {checks['unknown_category']:,}")